# 02 — Skeleton Extraction: ShuttleSet (GDINO-guided)

Extracts dual-player skeletons for ShuttleSet using **Grounding DINO + YOLOv8-Pose**.
GDINO acts as a player-specific detector (prompt: `"badminton player ."`) providing
bounding-box priors; YOLO keypoints are only kept for detections with IoU ≥ 0.25
against a GDINO prior. This eliminates chair-umpire / ball-kid contamination.

**Output format:** `(2, T, 34)` per-shot `.npy` files  
- `dim 0` = [x, y]  
- `dim 1` = T=16 frames centred on `frame_num`  
- `dim 2` = 34 joints: nodes 0–16 = P0 (top-court / smaller Y), 17–33 = P1 (bottom-court)
- Hitter-first reorder applied using `player_location_y` from ShuttleSet CSV records

**Saved to:** `datasets_preprocessing/ShuttleSet/skeletons_gdino/{match_id}/r{rally:04d}_b{ball_round:04d}.npy`

**Steps:**
1. (Optional) Cleanup old GDINO skeletons
2. GDINO model setup
3. Load shot records
4. GDINO-guided extraction loop
5. Contamination scan + Y-sort verification
6. Dataset loading test

In [ ]:
import os, sys, zipfile

# ── Colab setup (skipped when running locally) ────────────────────────────────
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    ZIP_PATH    = '/content/drive/MyDrive/FineBadminton/baddiev2_colab.zip'
    PROJECT_PATH = '/content/Baddiev2'

    if not os.path.exists(PROJECT_PATH):
        print("Extracting project files...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(PROJECT_PATH)
        print("Done.")
    else:
        print("Project already extracted.")

    os.chdir(os.path.join(PROJECT_PATH, 'notebooks'))
    sys.path.insert(0, PROJECT_PATH)
    print(f"CWD: {os.getcwd()}")

    # Install yt-dlp (yt-dlp is more maintained than youtube-dl)
    os.system("pip install -q yt-dlp")
else:
    print("Local run")

In [ ]:
## ── Unzip pre-extracted frames from Google Drive (Colab only) ────────────────
## Upload these zips to Drive before running:
##   fuzhou_frames.zip    → frames for Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_Finals
##   ginting_frames.zip   → frames for Anthony_Sinisuka_GINTING_Viktor_AXELSEN_...
##
## Drive path: MyDrive/FineBadminton/fuzhou_frames.zip  (same folder as baddiev2_colab.zip)

if IN_COLAB:
    import os, zipfile
    from pathlib import Path

    DRIVE_DIR   = Path('/content/drive/MyDrive/FineBadminton')
    FRAMES_ROOT = Path('/content/Baddiev2/datasets_preprocessing/shuttleset_frames')
    FRAMES_ROOT.mkdir(parents=True, exist_ok=True)

    frame_zips = {
        'fuzhou_frames.zip':  'Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_Finals',
        'ginting_frames.zip': 'Anthony_Sinisuka_GINTING_Viktor_AXELSEN _Indonesia_Masters_2020_SemiFinals',
    }

    for zip_name, match_id in frame_zips.items():
        dest_dir = FRAMES_ROOT / match_id
        zip_path = DRIVE_DIR / zip_name

        if dest_dir.exists() and len(list(dest_dir.glob('*.jpg'))) > 100:
            print(f"[SKIP] {match_id[:50]} — already unzipped ({len(list(dest_dir.glob('*.jpg')))} frames)")
            continue

        if not zip_path.exists():
            print(f"[WARN] {zip_name} not found in Drive at {zip_path} — skipping")
            continue

        print(f"Unzipping {zip_name} → {dest_dir} ...")
        dest_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(dest_dir)
        n = len(list(dest_dir.glob('*.jpg')))
        print(f"  Done — {n} frames extracted")
else:
    print("Local run — frames already on disk, skipping unzip")

In [ ]:
!pip install -q tqdm transformers timm pillow

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import cv2
import torch
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

from src.config import (
    PROJECT_ROOT,
    SS_FRAMES, SS_OUTPUTS, SS_SKELETONS_GDINO,
    get_config,
)
from src.data.dataset import _reorder_hitter_first_by_location

cfg = get_config()
HALF = cfg.data.shot_window // 2   # 8 frames before/after hit frame
print(f"Shot window T={cfg.data.shot_window}, half={HALF}")
print(f"GDINO skeletons → {SS_SKELETONS_GDINO}")

## 0. (Optional) Cleanup

Delete previously extracted GDINO skeletons to force a full re-extraction.

In [ ]:
import shutil
FORCE_REEXTRACT = False  # ← set True to wipe and re-extract

if FORCE_REEXTRACT and SS_SKELETONS_GDINO.exists():
    existing = list(SS_SKELETONS_GDINO.rglob('*.npy'))
    for f in existing:
        f.unlink()
    print(f"Deleted {len(existing)} GDINO skeleton files")
else:
    n = len(list(SS_SKELETONS_GDINO.rglob('*.npy'))) if SS_SKELETONS_GDINO.exists() else 0
    print(f"Skipping cleanup — {n} existing GDINO skeletons retained")

## 1. MVP Mode

Set `MVP_MODE = True` to extract only a few matches for a quick pipeline check.

**Timing estimates (per shot = 16 frames):**
- CPU: ~8–15 s/frame → ~2–4 min/shot → not practical for full dataset
- GPU T4: ~0.3–0.5 s/frame → ~5–8 s/shot → ~1 h for the An_Se_Young match (663 shots)

In [ ]:
## ── Frame download cell (COMMENTED OUT) ──────────────────────────────────────
## Frames are already extracted locally as JPG and uploaded to Google Drive.
## Use the "Unzip frames from Drive" cell below instead.
## ─────────────────────────────────────────────────────────────────────────────
#
# import csv, subprocess, shutil
#
# if IN_COLAB:
#     match_csv_path = PROJECT_ROOT / 'datasets' / 'ShuttleSet' / 'set' / 'match.csv'
#     url_map = {}
#     with open(match_csv_path) as f:
#         for row in csv.DictReader(f):
#             url_map[row['video']] = row['url']
#     print(f"Loaded {len(url_map)} YouTube URLs from match.csv")
#
#     from src.config import SS_OUTPUTS, SS_FRAMES as _SS_FRAMES
#     json_match_ids = [jf.stem for jf in sorted(SS_OUTPUTS.glob('*.json'))]
#     target_ids = json_match_ids[:MVP_MATCHES] if MVP_MODE else json_match_ids
#
#     _SS_FRAMES.mkdir(parents=True, exist_ok=True)
#     for match_id in target_ids:
#         frame_dir = _SS_FRAMES / match_id
#         if frame_dir.exists() and len(list(frame_dir.glob('*.jpg'))) > 100:
#             print(f"  [SKIP] {match_id[:50]} — frames already present")
#             continue
#         yt_url = url_map.get(match_id)
#         if not yt_url:
#             print(f"  [SKIP] {match_id[:50]} — no YouTube URL")
#             continue
#         # ... (download + ffmpeg extract) ...
# else:
#     print("Local run — skipping download")

print("Download cell skipped — frames loaded from Drive zips below.")

In [ ]:
MVP_MODE    = True   # ← False to process all matches
MVP_MATCHES = 1      # matches to process in MVP mode (used only if MVP_MATCH_IDS is empty)

# Explicit match IDs to target (overrides alphabetical first-N selection).
# Set to [] to fall back to the first MVP_MATCHES alphabetically.
MVP_MATCH_IDS = [
    'Kento_MOMOTA_CHOU_Tien_Chen_Fuzhou_Open_2019_Finals',
    'Anthony_Sinisuka_GINTING_Viktor_AXELSEN _Indonesia_Masters_2020_SemiFinals',
]

if MVP_MODE:
    if MVP_MATCH_IDS:
        print(f"MVP mode ON — explicit targets: {MVP_MATCH_IDS}")
    else:
        print(f"MVP mode ON — extracting first {MVP_MATCHES} match(es) alphabetically")
else:
    print("Full extraction mode — processing all matches")

## 2. Grounding DINO Setup

Load GDINO-tiny and define per-frame extraction helpers.

**Pipeline per frame:**
1. GDINO detects `"badminton player"` → bounding boxes (priors)
2. YOLOv8-Pose detects all persons → keypoints
3. Keep only YOLO detections with IoU ≥ 0.25 vs a GDINO prior
4. Y-sort the two kept players → P0 (top-court), P1 (bottom-court)

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image as _PIL
from ultralytics import YOLO

_DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_MODEL_ID     = 'IDEA-Research/grounding-dino-tiny'
_BOX_THRESH   = 0.30
_TXT_THRESH   = 0.25
_IOU_THRESH   = 0.25    # min IoU vs GDINO prior to keep a YOLO detection
_GDINO_PROMPT = 'badminton player .'   # specific prompt — avoids umpire / ball kids

print(f"Loading GDINO-tiny on {_DEVICE}...")
_gdino_proc  = AutoProcessor.from_pretrained(_MODEL_ID)
_gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(_MODEL_ID).to(_DEVICE)
_gdino_model.eval()
print("  GDINO ready")

_yolo = YOLO('yolov8s-pose.pt')   # swap to yolov8x-pose.pt for best accuracy on GPU
print("  YOLOv8s-pose ready")


# ── helpers ───────────────────────────────────────────────────────────────────

@torch.no_grad()
def _gdino_detect(bgr):
    """Run GDINO on a BGR frame → (boxes, scores) arrays in pixel coords."""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    H, W = bgr.shape[:2]
    inp  = _gdino_proc(
        images=_PIL.fromarray(rgb), text=_GDINO_PROMPT, return_tensors='pt'
    ).to(_DEVICE)
    out  = _gdino_model(**inp)
    res  = _gdino_proc.post_process_grounded_object_detection(
        outputs=out, input_ids=inp['input_ids'],
        threshold=_BOX_THRESH, text_threshold=_TXT_THRESH,
        target_sizes=[(H, W)]
    )[0]
    return res['boxes'].cpu().numpy(), res['scores'].cpu().numpy()


def _iou(b1, b2):
    """Intersection-over-Union of two [x1,y1,x2,y2] boxes."""
    ix1, iy1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    ix2, iy2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter    = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    union    = ((b1[2]-b1[0])*(b1[3]-b1[1]) +
                (b2[2]-b2[0])*(b2[3]-b2[1]) - inter + 1e-9)
    return inter / union


def _extract_gdino_frame(bgr):
    """
    Extract a (2, 34) skeleton from one BGR frame using GDINO+YOLO.
    Returns zeros on complete detection failure.
    """
    g_boxes, _ = _gdino_detect(bgr)

    res = _yolo(bgr, verbose=False)[0]
    out = np.zeros((2, 34), dtype=np.float32)
    if res is None or res.keypoints is None or res.boxes is None:
        return out

    y_boxes = res.boxes.xyxy.cpu().numpy()   # (N, 4)
    y_cls   = res.boxes.cls.cpu().numpy()
    y_kpts  = res.keypoints.xy.cpu().numpy() # (N, 17, 2)
    y_confs = res.boxes.conf.cpu().numpy()

    # Keep YOLO person detections that overlap with any GDINO prior
    kept_kpts, kept_confs, kept_cy = [], [], []
    for i, (yb, yc, yconf) in enumerate(zip(y_boxes, y_cls, y_confs)):
        if int(yc) != 0:   # persons only
            continue
        max_iou = max((_iou(yb, gb) for gb in g_boxes), default=0.0) if len(g_boxes) else 0.0
        if max_iou >= _IOU_THRESH:
            kept_kpts.append(y_kpts[i])
            kept_confs.append(float(yconf))
            kept_cy.append(float(y_kpts[i][:, 1].mean()))

    # Fallback: plain top-2 YOLO if GDINO filtering left fewer than 2 players
    if len(kept_kpts) < 2:
        pmask = y_cls == 0
        if pmask.sum() >= 2:
            order     = np.argsort(y_confs[pmask])[::-1][:2]
            pkpts     = y_kpts[pmask]
            kept_kpts = [pkpts[j] for j in order]
            kept_cy   = [float(kept_kpts[j][:, 1].mean()) for j in range(2)]
        else:
            return out   # can't extract 2 players

    # If more than 2 passed the filter, keep the 2 most confident
    if len(kept_kpts) > 2:
        order     = np.argsort(kept_confs)[::-1][:2]
        kept_kpts = [kept_kpts[j] for j in order]
        kept_cy   = [kept_cy[j]   for j in order]

    # Y-sort: smaller Y centroid → P0 (top-court)
    ys = np.argsort(kept_cy)
    p0, p1 = kept_kpts[ys[0]], kept_kpts[ys[1]]
    out[0, :17] = p0[:, 0];  out[1, :17] = p0[:, 1]
    out[0, 17:] = p1[:, 0];  out[1, 17:] = p1[:, 1]
    return out


print("\nGDINO + YOLO helpers ready.")

## 3. Load Shot Records

In [ ]:
# Load all shot records grouped by match_id
match_records = {}   # match_id → list of records
for json_file in sorted(SS_OUTPUTS.glob('*.json')):
    if json_file.name == 'pipeline_summary.json':
        continue
    with open(json_file) as f:
        records = json.load(f)
    if records:
        match_id = records[0].get('match_id', json_file.stem)
        match_records[match_id] = records

print(f"Matches with JSON records: {len(match_records)}")

match_ids = sorted(match_records.keys())
if MVP_MODE:
    if MVP_MATCH_IDS:
        match_ids = [mid for mid in MVP_MATCH_IDS if mid in match_records]
        missing = [mid for mid in MVP_MATCH_IDS if mid not in match_records]
        if missing:
            print(f"[WARN] MVP_MATCH_IDS not found in outputs: {missing}")
    else:
        match_ids = match_ids[:MVP_MATCHES]
    print(f"MVP: {match_ids}")

for mid in match_ids:
    n_frames = len(list((SS_FRAMES / mid).glob('*.png'))) if (SS_FRAMES / mid).exists() else 0
    n_shots  = len(match_records[mid])
    already  = len(list((SS_SKELETONS_GDINO / mid).glob('*.npy'))) if (SS_SKELETONS_GDINO / mid).exists() else 0
    print(f"  {mid[:60]}")
    print(f"    frames={n_frames}  shots={n_shots}  already_extracted={already}")

## 4. GDINO-Guided Extraction

For each shot:  
1. Collect T=16 frames centred on `frame_num` (forward-fill on missing frames)  
2. Run `_extract_gdino_frame` on each frame → `(2, 34)`  
3. Forward-fill on complete detection failures  
4. Apply hitter-first reorder using `player_location_y`  
5. Save as `(2, T, 34)` `.npy`  

Already-extracted shots are skipped automatically.

In [ ]:
from collections import defaultdict

SS_SKELETONS_GDINO.mkdir(parents=True, exist_ok=True)

for match_id in tqdm(match_ids, desc='Matches'):
    match_frame_dir = SS_FRAMES / match_id
    match_skel_dir  = SS_SKELETONS_GDINO / match_id
    match_skel_dir.mkdir(parents=True, exist_ok=True)

    if not match_frame_dir.exists():
        print(f"  [SKIP] No frames: {match_id}")
        continue

    # Build frame_num → path index (jpg takes priority over png)
    frame_index = {}
    for ext in ('*.jpg', '*.png'):
        for fp in match_frame_dir.glob(ext):
            try:
                fnum = int(fp.stem.replace('frame_', ''))
                if fnum not in frame_index:
                    frame_index[fnum] = fp
            except ValueError:
                pass

    if not frame_index:
        print(f"  [SKIP] Empty frame dir: {match_id}")
        continue

    # Group records by rally integer
    rally_groups = defaultdict(list)
    for rec in match_records[match_id]:
        rally_groups[rec['rally']].append(rec)

    saved = skipped = 0

    for rally_int, records in tqdm(sorted(rally_groups.items()),
                                   desc=f'  {match_id[:35]}', leave=False):
        rally_file = match_skel_dir / f"r{rally_int:04d}.npy"

        if rally_file.exists():
            saved += 1
            continue

        # All shots in a rally share the same rally_frame_start / rally_frame_end
        frame_start = records[0].get('rally_frame_start')
        frame_end   = records[0].get('rally_frame_end')
        if frame_start is None or frame_end is None:
            skipped += 1
            continue

        T_rally  = frame_end - frame_start + 1
        skel_arr = np.zeros((2, T_rally, 34), dtype=np.float32)
        n_fallback = 0

        for t in range(T_rally):
            fn  = frame_start + t
            fp  = frame_index.get(fn)
            bgr = cv2.imread(str(fp)) if fp else None
            if bgr is None:
                if t > 0:
                    skel_arr[:, t, :] = skel_arr[:, t - 1, :]
                n_fallback += 1
                continue
            sf = _extract_gdino_frame(bgr)
            if sf.sum() == 0 and t > 0:
                sf = skel_arr[:, t - 1, :]
                n_fallback += 1
            skel_arr[:, t, :] = sf

        np.save(rally_file, skel_arr)
        saved += 1

    total = len(list(match_skel_dir.glob('*.npy')))
    print(f"  {match_id[:50]}: rallies saved={saved}, skipped={skipped}, total_npy={total}")

grand_total = len(list(SS_SKELETONS_GDINO.rglob('*.npy')))
print(f"\nTotal GDINO rally skeletons: {grand_total}")

## 5. Contamination Scan + Y-Sort Verification

Checks for residual contamination (chair umpire detected as P0).
- `UMPIRE_X_THRESH` = 300px (≈15% of 1920px frame width — umpire sits near left edge)
- Y-sort: P0 mean Y should always be smaller (higher on screen) than P1

In [ ]:
UMPIRE_X_THRESH = 300   # pixels; umpire sits near left edge of 1920px frame

skel_files = sorted(SS_SKELETONS_GDINO.rglob('*.npy'))
print(f"GDINO skeletons found: {len(skel_files)}")

if not skel_files:
    print("  No skeletons yet — run extraction cell first.")
else:
    y_ok = y_fail = 0
    contaminated = 0
    p0x_vals = []

    for f in skel_files:
        skel = np.load(f)               # (2, T, 34)
        p0_y = skel[1, :, :17].mean()
        p1_y = skel[1, :, 17:].mean()
        p0_x = skel[0, :, :17].mean()
        p0x_vals.append(p0_x)

        if p0_y < p1_y:
            y_ok += 1
        else:
            y_fail += 1

        if p0_x < UMPIRE_X_THRESH:
            contaminated += 1

    total = y_ok + y_fail
    print(f"\nY-sort  : {y_ok}/{total} OK ({100*y_ok/total:.1f}%),  "
          f"{y_fail} FAIL")
    print(f"Umpire  : {contaminated}/{total} contaminated "
          f"({100*contaminated/total:.1f}%)  [thresh={UMPIRE_X_THRESH}px]")
    print(f"P0 mean-X  — median={np.median(p0x_vals):.0f}px, "
          f"min={np.min(p0x_vals):.0f}px, max={np.max(p0x_vals):.0f}px")

    # Histogram of P0 x centroid
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.hist(p0x_vals, bins=60, color='#60a5fa', alpha=0.8)
    ax.axvline(UMPIRE_X_THRESH, color='#ef4444', lw=2,
               label=f'Umpire threshold ({UMPIRE_X_THRESH}px)')
    ax.set_xlabel('P0 mean X centroid (pixels)')
    ax.set_ylabel('Shots')
    ax.set_title('GDINO extraction — P0 X centroid distribution\n'
                 '(values < threshold indicate chair umpire contamination)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5b. Visual Verification

Overlay skeletons on actual frames for a random selection of shots.

In [ ]:
COCO_EDGES = [
    (0,1),(0,2),(1,3),(2,4),(5,6),(5,7),(7,9),
    (6,8),(8,10),(5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]
P_KP  = [(0,255,255), (255,0,255)]   # cyan / magenta (BGR)
P_LMB = [(0,150,150), (150,0,150)]

def _draw_player(img, skel_xy, p):
    off = p * 17
    xs, ys = skel_xy[0, off:off+17], skel_xy[1, off:off+17]
    for j1, j2 in COCO_EDGES:
        x1,y1,x2,y2 = int(xs[j1]),int(ys[j1]),int(xs[j2]),int(ys[j2])
        if x1>0 and y1>0 and x2>0 and y2>0:
            cv2.line(img,(x1,y1),(x2,y2),P_LMB[p],2)
    for k in range(17):
        x,y = int(xs[k]),int(ys[k])
        if x>0 or y>0:
            cv2.circle(img,(x,y),5,P_KP[p],-1)
    ok = (xs>0)&(ys>0)
    if ok.any():
        cx,cy = int(xs[ok].mean()),int(ys[ok].mean())
        cv2.putText(img,f'P{p}',(cx-10,cy-14),
                    cv2.FONT_HERSHEY_SIMPLEX,0.7,P_KP[p],2)

# Pick a few shots to visualise
SHOW_SHOTS = 6
all_shot_files = sorted(SS_SKELETONS_GDINO.rglob('*.npy'))

if not all_shot_files:
    print("No GDINO skeletons yet.")
else:
    # Spread evenly across extracted shots
    idxs = np.linspace(0, len(all_shot_files)-1, SHOW_SHOTS, dtype=int)
    sample_files = [all_shot_files[i] for i in idxs]

    fig, axes = plt.subplots(1, SHOW_SHOTS, figsize=(4*SHOW_SHOTS, 5))
    if SHOW_SHOTS == 1:
        axes = [axes]

    for ax, sf in zip(axes, sample_files):
        # Recover match_id / rally / ball from path
        ball_name = sf.stem                        # e.g. r0001_b0003
        match_id  = sf.parent.name
        parts     = ball_name.split('_')
        rally     = int(parts[0][1:])              # strip leading 'r'
        ball_rnd  = int(parts[1][1:])              # strip leading 'b'

        skel = np.load(sf)                         # (2, T, 34)
        t_mid = skel.shape[1] // 2

        # Find the corresponding frame on disk (centre of the shot window)
        # Read JSON to get frame_num
        json_file = next(iter(SS_OUTPUTS.glob(f'{match_id}*.json')), None)
        frame_num = None
        if json_file:
            with open(json_file) as jf:
                for rec in json.load(jf):
                    if rec.get('rally')==rally and rec.get('ball_round')==ball_rnd:
                        frame_num = rec.get('frame_num')
                        break

        img = None
        if frame_num is not None:
            fp = SS_FRAMES / match_id / f"frame_{frame_num:06d}.png"
            if fp.exists():
                img = cv2.imread(str(fp))

        if img is None:
            ax.text(0.5,0.5,'frame\nmissing',ha='center',va='center')
            ax.axis('off')
            continue

        for p in range(2):
            _draw_player(img, skel[:, t_mid, :], p)

        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{match_id[:20]}\nR{rally} B{ball_rnd} t={t_mid}', fontsize=7)
        ax.axis('off')

    plt.suptitle(
        'GDINO-guided extraction — centre frame of each shot\n'
        'Cyan = P0 (top-court, smaller Y)   Magenta = P1 (bottom-court)',
        fontsize=10
    )
    plt.tight_layout()
    plt.show()

## 6. Dataset Loading Test

Verify `ShuttleSetDataset` can load GDINO skeletons through the standard pipeline.

In [ ]:
from src.data.dataset import ShuttleSetDataset

# Pass gdino=True (or skeleton_dir override) once ShuttleSetDataset supports it
# For now, verify the npy files load correctly with the expected shape
from src.config import SS_SKELETONS_GDINO
from src.data.feature_eng import FeatureEngineer

eng = FeatureEngineer(feature_layer='L2')

gdino_files = sorted(SS_SKELETONS_GDINO.rglob('*.npy'))
print(f"GDINO skeleton files: {len(gdino_files)}")

if gdino_files:
    sample = np.load(gdino_files[0])
    print(f"  Raw shape : {sample.shape}  (expected (2, {cfg.data.shot_window}, 34))")
    feats = eng.compute(sample)
    print(f"  L2 features shape: {feats.shape}  (expected (9, {cfg.data.shot_window}, 34))")
    print(f"  Non-zero elements: {(feats != 0).sum()} / {feats.size}")
    print(f"  P0 mean-Y: {sample[1, :, :17].mean():.1f}  P1 mean-Y: {sample[1, :, 17:].mean():.1f}")

# Standard dataset (still uses original skeletons dir — update when all matches extracted)
ss_ds = ShuttleSetDataset(feature_layer='L2')
print(f"\nShuttleSetDataset: {len(ss_ds)} samples")
if len(ss_ds) > 0:
    x, st = ss_ds[0]
    print(f"  Sample shape: {x.shape}, shot_type idx: {st}")

In [ ]:
## ── Save extracted skeletons to Google Drive (Colab only) ────────────────────
## Run this after extraction to persist results before the session ends.
## Saves to: MyDrive/FineBadminton/shuttleset_skeletons_gdino.zip

if IN_COLAB:
    import shutil, os
    from pathlib import Path

    DRIVE_DIR  = Path('/content/drive/MyDrive/FineBadminton')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    skel_dir   = SS_SKELETONS_GDINO
    out_zip    = DRIVE_DIR / 'shuttleset_skeletons_gdino.zip'

    n_files = len(list(skel_dir.rglob('*.npy'))) if skel_dir.exists() else 0
    print(f"Skeletons to archive: {n_files} .npy files from {skel_dir}")

    if n_files == 0:
        print("[WARN] No skeletons found — run extraction first.")
    else:
        print(f"Zipping to {out_zip} ...")
        shutil.make_archive(str(out_zip).replace('.zip', ''), 'zip', skel_dir.parent, skel_dir.name)
        size_mb = out_zip.stat().st_size / 1e6
        print(f"Saved: {out_zip} ({size_mb:.1f} MB, {n_files} skeletons)")
else:
    print("Local run — skeletons already on disk, no Drive save needed")